# Kapitel 6 – Stufe 2: Hypothesentestung
**Masterarbeit | Kapitel 6.2**  
Autor: Ayumi Nojima | April 2026

---
- **H1** – ANOVA: Unterschiede im Exposure-Index zwischen Hauptgruppen
- **H2** – Random Forest: Fähigkeitsmerkmale als Prädiktoren für E^sub_j
- **H3** – OLS-Regression: Zusammenhang E_j und ΔBFS_j

**Hypothesen (vgl. Kap. 4):**
- H1: E_j unterscheidet sich signifikant zwischen CH-ISCO-Hauptgruppen 1–3
- H2: Kognitive/sprachliche Fähigkeiten sind stärkste Prädiktoren für E^sub_j
- H3: Höherer E_j geht mit stärkerer Beschäftigungsveränderung einher (Displacement-Effekt)


## 0. Konfiguration und Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
import statsmodels.api as sm
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings("ignore")

# ── Pfade ──────────────────────────────────────────────────────────────────
_cwd = Path.cwd()
REPO_ROOT = _cwd.parent if (_cwd / "..").resolve().joinpath("data").exists() else _cwd
if not (REPO_ROOT / "data").exists():
    REPO_ROOT = Path.cwd()

PROCESSED = REPO_ROOT / "data" / "processed"
ANALYSIS  = PROCESSED / "analysis_prep"
OUTPUT    = REPO_ROOT / "data" / "output"
FIGURES   = OUTPUT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

BFS_BASE_YEAR   = 2022
BFS_TARGET_YEAR = 2024

# ── Daten laden ────────────────────────────────────────────────────────────
final         = pd.read_csv(ANALYSIS / "final_sample_clustered.csv")
onet_pivot    = pd.read_csv(PROCESSED / "onet_pivot.csv")

h1 = pd.read_csv(OUTPUT / "dataset_H1.csv")
h2 = pd.read_csv(OUTPUT / "dataset_H2.csv")
h3 = pd.read_csv(OUTPUT / "dataset_H3.csv")

hg_labels = {1: "HG 1 – Führungskräfte", 2: "HG 2 – Akademisch", 3: "HG 3 – Techniker"}
colors     = {1: "#2196F3", 2: "#4CAF50", 3: "#FF9800"}

print(f"H1-Datensatz: {len(h1)} Berufe")
print(f"H2-Datensatz: {len(h2)} Berufe × {h2.shape[1]-2} Prädiktoren")
print(f"H3-Datensatz: {len(h3)} Berufe ({h3['delta_bfs'].notna().sum()} mit ΔBFS_j)")
print("Konfiguration geladen ✓")


---
## H1 – ANOVA: Gruppenunterschiede nach CH-ISCO-Hauptgruppe

**Vorgehen:** Einfaktorielle ANOVA mit E_j als abhängiger Variable und  
Hauptgruppe als Faktor. Bei signifikantem F-Test: Post-hoc Tukey-HSD.  
Voraussetzungsprüfung: Levene-Test (Varianzhomogenität), Shapiro-Wilk (Normalität).


In [ ]:
# ── Voraussetzungsprüfung ──────────────────────────────────────────────────
groups = [h1[h1["main_group"] == hg]["E_j"].dropna().values for hg in [1, 2, 3]]

# Levene-Test (Varianzhomogenität)
levene_stat, levene_p = stats.levene(*groups)
print(f"Levene-Test: F={levene_stat:.3f}, p={levene_p:.4f} "
      f"({'✓ Varianzhomogenität' if levene_p > 0.05 else '⚠ Varianzinhomogenität'})")

# Shapiro-Wilk je Gruppe
print("\nShapiro-Wilk Normalitätstest:")
for hg, g in zip([1,2,3], groups):
    stat, p = stats.shapiro(g)
    print(f"  {hg_labels[hg]}: W={stat:.3f}, p={p:.4f} "
          f"({'✓ Normal' if p > 0.05 else '⚠ Nicht normal'})")


In [ ]:
# ── Einfaktorielle ANOVA ──────────────────────────────────────────────────
f_stat, p_value = stats.f_oneway(*groups)
print(f"ANOVA: F={f_stat:.3f}, p={p_value:.6f}")
print(f"Ergebnis: {'H1 wird unterstützt – signifikante Gruppenunterschiede' if p_value < 0.05 else 'Kein signifikanter Unterschied'} (α=0.05)")

# Eta-Quadrat (Effektstärke)
grand_mean = np.concatenate(groups).mean()
ss_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in groups)
ss_total   = sum(((g - grand_mean)**2).sum() for g in groups)
eta_sq     = ss_between / ss_total
print(f"Eta² = {eta_sq:.3f} ({'klein' if eta_sq < 0.06 else 'mittel' if eta_sq < 0.14 else 'gross'})")


In [ ]:
# ── Post-hoc: Tukey-HSD ───────────────────────────────────────────────────
from statsmodels.stats.multicomp import pairwise_tukeyhsd

tukey = pairwise_tukeyhsd(
    endog=h1["E_j"].dropna(),
    groups=h1.loc[h1["E_j"].notna(), "main_group"],
    alpha=0.05
)
print("\nTukey-HSD Post-hoc Test:")
print(tukey)


In [ ]:
# ── Visualisierung H1 ─────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("H1: LLM-Exposure nach CH-ISCO-Hauptgruppe", fontsize=13, fontweight="bold")

# Boxplot
data_plot = [h1[h1["main_group"] == hg]["E_j"].dropna() for hg in [1,2,3]]
bp = ax1.boxplot(data_plot, labels=[hg_labels[h] for h in [1,2,3]], patch_artist=True,
                 medianprops={"color": "black", "linewidth": 2})
for patch, hg in zip(bp["boxes"], [1,2,3]):
    patch.set_facecolor(colors[hg]); patch.set_alpha(0.75)
ax1.set_ylabel("E_j [0,1]"); ax1.set_title(f"ANOVA: F={f_stat:.2f}, p={p_value:.4f}")
ax1.set_ylim(0, 1)

# Mittelwerte mit CI
means = [g.mean() for g in data_plot]
cis   = [stats.sem(g) * stats.t.ppf(0.975, len(g)-1) for g in data_plot]
ax2.bar([1,2,3], means, yerr=cis, color=[colors[h] for h in [1,2,3]],
        alpha=0.75, capsize=5, edgecolor="black", linewidth=0.8)
ax2.set_xticks([1,2,3]); ax2.set_xticklabels([hg_labels[h] for h in [1,2,3]], fontsize=9)
ax2.set_ylabel("Mittlerer E_j"); ax2.set_title("Mittelwerte ± 95%-KI"); ax2.set_ylim(0, 1)

plt.tight_layout()
plt.savefig(FIGURES / "6_2_H1_anova.png", dpi=150, bbox_inches="tight")
plt.show()


---
## H2 – Random Forest: Fähigkeitsprädiktoren für E^sub_j

**Vorgehen:** Random Forest Regressor mit 5-facher Kreuzvalidierung.  
Feature Importance via permutation importance (Breiman, 2001).  
Ziel: Identifikation der Fähigkeitsdimensionen mit stärkstem Einfluss auf Substitutionsdruck.


In [ ]:
# ── Datenvorbereitung H2 ──────────────────────────────────────────────────
feature_cols = [c for c in h2.columns if c not in ["soc_code", "E_sub_j"]]
X_h2 = h2[feature_cols].fillna(0).values
y_h2 = h2["E_sub_j"].values

print(f"Features: {len(feature_cols)}")
print(f"Sample:   {len(y_h2)} Berufe")
print(f"E^sub_j:  Mean={y_h2.mean():.3f} | Std={y_h2.std():.3f}")


In [ ]:
# ── Random Forest Training + CV ───────────────────────────────────────────
rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=3,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

cv_scores = cross_val_score(rf, X_h2, y_h2, cv=5, scoring="r2")
print(f"5-fache CV R²: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
print(f"CV-Scores: {cv_scores.round(3)}")

# Volles Modell für Feature Importance
rf.fit(X_h2, y_h2)
print(f"\nIn-Sample R²: {rf.score(X_h2, y_h2):.3f}")


In [ ]:
# ── Feature Importance (Permutation) ──────────────────────────────────────
perm_imp = permutation_importance(rf, X_h2, y_h2, n_repeats=30, random_state=42, n_jobs=-1)

importance_df = pd.DataFrame({
    "feature":    feature_cols,
    "importance": perm_imp.importances_mean,
    "std":        perm_imp.importances_std
}).sort_values("importance", ascending=False).reset_index(drop=True)

print("=== Top 20 Feature Importance (Permutation) ===")
print(importance_df.head(20).to_string(index=False))

importance_df.to_csv(ANALYSIS / "H2_feature_importance.csv", index=False)
print("\nGespeichert → data/processed/analysis_prep/H2_feature_importance.csv ✓")


In [ ]:
# ── Visualisierung Top 20 ─────────────────────────────────────────────────
top20 = importance_df.head(20)

fig, ax = plt.subplots(figsize=(10, 8))
colors_fi = ["#E53935" if imp > 0 else "#90A4AE" for imp in top20["importance"]]
bars = ax.barh(top20["feature"], top20["importance"],
               xerr=top20["std"], color=colors_fi, alpha=0.85,
               capsize=3, edgecolor="white", linewidth=0.5)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Permutation Importance (Δ R²)")
ax.set_title(f"H2: Top-20 Fähigkeitsprädiktoren für E^sub_j\n(Random Forest, 5-CV R²={cv_scores.mean():.3f})", fontsize=12)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(FIGURES / "6_2_H2_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()


---
## H3 – OLS-Regression: E_j und ΔBFS_j

**Vorgehen:** OLS-Regression mit ΔBFS_j als abhängiger Variable und E_j sowie  
Kontrollvariablen (Hauptgruppe, Qualifikationsniveau) als Prädiktoren.  
Robustheitsprüfung: Regression ohne Ausreisser (IQR-geflaggt).  
Quelle: Kläui & Siegenthaler (2025) als Orientierung für Effektgrössen.


In [ ]:
# ── Datenvorbereitung H3 ──────────────────────────────────────────────────
h3_clean = h3.dropna(subset=["E_j", "delta_bfs"]).copy()
print(f"H3-Sample (ohne NaN): {len(h3_clean)} Berufe")
print(f"  davon Ausreisser:   {h3_clean['is_outlier'].sum()}")
print(f"  Hauptanalyse:       {(~h3_clean['is_outlier']).sum()} Berufe (ohne Ausreisser)")


In [ ]:
# ── Modell 1: Bivariat E_j → ΔBFS_j ──────────────────────────────────────
h3_main = h3_clean[~h3_clean["is_outlier"]].copy()

model1 = smf.ols("delta_bfs ~ E_j", data=h3_main).fit(cov_type="HC3")
print("=== Modell 1: Bivariat (ohne Ausreisser) ===")
print(model1.summary2().tables[1].round(3))
print(f"R² = {model1.rsquared:.3f} | Adj. R² = {model1.rsquared_adj:.3f}")


In [ ]:
# ── Modell 2: Mit Kontrollvariablen ───────────────────────────────────────
# Hauptgruppe als kategoriale Kontrollvariable (HG 2 = Referenz)
h3_main["hg_cat"] = h3_main["main_group"].astype("category")

model2 = smf.ols("delta_bfs ~ E_j + C(main_group, Treatment(2))", data=h3_main).fit(cov_type="HC3")
print("=== Modell 2: Mit Hauptgruppen-Kontrolle ===")
print(model2.summary2().tables[1].round(3))
print(f"R² = {model2.rsquared:.3f} | Adj. R² = {model2.rsquared_adj:.3f}")


In [ ]:
# ── Modell-Vergleich ───────────────────────────────────────────────────────
from statsmodels.stats.anova import anova_lm

comparison = pd.DataFrame({
    "Modell":    ["M1: Bivariat", "M2: + Hauptgruppe"],
    "N":         [model1.nobs, model2.nobs],
    "R²":        [model1.rsquared, model2.rsquared],
    "Adj. R²":   [model1.rsquared_adj, model2.rsquared_adj],
    "AIC":       [model1.aic, model2.aic],
    "E_j Koeff": [model1.params["E_j"], model2.params["E_j"]],
    "E_j p":     [model1.pvalues["E_j"], model2.pvalues["E_j"]],
}).round(3)
print(comparison.to_string(index=False))


In [ ]:
# ── Robustheitsprüfung: inkl. Ausreisser ──────────────────────────────────
model_robust = smf.ols("delta_bfs ~ E_j + C(main_group, Treatment(2))", data=h3_clean).fit(cov_type="HC3")
print("=== Robustheitsprüfung: Alle Berufe inkl. Ausreisser ===")
print(f"E_j: β={model_robust.params['E_j']:.3f}, p={model_robust.pvalues['E_j']:.4f}")
print(f"R²={model_robust.rsquared:.3f} (vs. {model2.rsquared:.3f} ohne Ausreisser)")


In [ ]:
# ── Visualisierung H3 ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("H3: E_j und Beschäftigungsveränderung ΔBFS_j", fontsize=13, fontweight="bold")

# Scatter mit Regressionslinie
ax = axes[0]
for hg in [1, 2, 3]:
    s = h3_main[h3_main["main_group"] == hg]
    ax.scatter(s["E_j"], s["delta_bfs"], color=colors[hg], alpha=0.6, s=50,
               label=hg_labels[hg], edgecolors="white", linewidth=0.3)
x_pred = np.linspace(h3_main["E_j"].min(), h3_main["E_j"].max(), 100)
y_pred = model2.params["Intercept"] + model2.params["E_j"] * x_pred
ax.plot(x_pred, y_pred, "k--", linewidth=2, label="OLS M2")
ax.axhline(0, color="gray", linewidth=0.8, linestyle=":")
ax.set_xlabel("E_j"); ax.set_ylabel("ΔBFS_j (%)"); ax.set_title("Scatter + OLS")
ax.legend(fontsize=8)

# Residuen vs. Fitted
ax = axes[1]
fitted   = model2.fittedvalues
residuals = model2.resid
ax.scatter(fitted, residuals, alpha=0.5, s=40, color="#5C6BC0", edgecolors="white", linewidth=0.3)
ax.axhline(0, color="red", linewidth=1, linestyle="--")
ax.set_xlabel("Fitted Values"); ax.set_ylabel("Residuen")
ax.set_title("Residuen vs. Fitted (M2)")

# Q-Q Plot
ax = axes[2]
stats.probplot(residuals, dist="norm", plot=ax)
ax.set_title("Q-Q Plot Residuen (M2)")

plt.tight_layout()
plt.savefig(FIGURES / "6_2_H3_regression.png", dpi=150, bbox_inches="tight")
plt.show()

# Ergebnisse speichern
results_h3 = pd.DataFrame({
    "model": ["M1_bivariat", "M2_kontrolliert", "M_robust"],
    "n": [model1.nobs, model2.nobs, model_robust.nobs],
    "r2": [model1.rsquared, model2.rsquared, model_robust.rsquared],
    "beta_Ej": [model1.params["E_j"], model2.params["E_j"], model_robust.params["E_j"]],
    "p_Ej": [model1.pvalues["E_j"], model2.pvalues["E_j"], model_robust.pvalues["E_j"]],
}).round(4)
results_h3.to_csv(ANALYSIS / "H3_regression_results.csv", index=False)
print("Gespeichert → data/processed/analysis_prep/H3_regression_results.csv ✓")
